# Day 23 · 错误分析与 Bad Case 归类

**配套讲义**: [`days/day-23.md`](../days/day-23.md) ｜ **本地可跑，不需要 GPU**

把失败样本自动聚类 + 关键词归类，输出 top 10 失败模式，并导出 `bad_cases.jsonl` —— 这份文件是 Day 26 构造 DPO 数据的原料。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w4.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 跑归类

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.eval.error_analysis",
                    "--in", "reports/eval_lora_raw.jsonl", "--out-dir", "reports/"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-2500:] or r.stderr[-2500:])

## 2. 扩业务词表（今天的核心动作）

关键词组越贴合你的业务，归类越准。把下面填成你们真实的话术。

In [ ]:
import sys; sys.path.insert(0, "..")
from src.eval.error_analysis import KEYWORD_GROUPS

for k, v in KEYWORD_GROUPS.items():
    print(f"{k:12s} {v if isinstance(v, list) else v}")

my_extra = {
    # "物流时效": ["几天到", "什么时候发货", "空运"],
    # "尺码争议": ["偏大", "偏小", "跟平时穿的一样吗"],
}
print("\n我补充的业务词表:", my_extra)

## 3. 出声读 10 条真实 bad case

这一步不能省。机器归类给你骨架，读样本给你血肉。

In [ ]:
import json
from pathlib import Path
p = Path("../reports/bad_cases.jsonl")
if p.exists():
    rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()]
    print(f"共 {len(rows)} 条，随机看 10 条：")
    import random
    for r in random.sample(rows, min(10, len(rows))):
        print("-" * 70)
        print("类型:", r.get("error_type"))
        print("问  :", str(r.get("question"))[:90])
        print("答  :", str(r.get("prediction", r.get("answer")))[:160])
else:
    print("先跑归类")

## 验收清单

- [ ] 错误分类表已产出，每类**至少 2 条典型样本**
- [ ] 能明确指出**下一轮该补哪三类数据**（不是「都补」）
- [ ] `bad_cases.jsonl` 已按 `error_type` 打标（Day 26 直接消费）
- [ ] 能指出哪几类失败其实同源（比如「漏信息」和「不当拒答」可能都是数据不足）

**卡住了？** 回看 [`days/day-23.md`](../days/day-23.md) 第五节「容易踩的坑」。

> **明天**：`days/day-24.md` —— 出评测报告 v1，W4 收官